# 🌱 Análise Exploratória — PlantVillage Dataset

**Projeto:** Edge AI para Detecção de Estresse Hídrico e Pragas com Redução de Overhead de Rede  
**Disciplina:** Inteligência Artificial — 7º Período CC Noite  
**Instituição:** Universidade Presbiteriana Mackenzie  
**Professor:** Prof. Dr. Ivan Carlos Alcântara de Oliveira  

---

## Integrantes

| Nome | RA |
|------|----|
| Gabriel Vieira de Sousa | 10410264 |
| Guilherme Rainho Geraldo | 10418251 |
| Guilherme Gomes Arantes Teles | 10364065 |

---

## Histórico de Alterações

| Data | Autor | Descrição |
|------|-------|-----------|
| 22/05/2026 | Guilherme Rainho | Criação do notebook e estrutura inicial |
| 22/05/2026 | Gabriel Vieira | Análise de distribuição de classes e RGB |
| 22/05/2026 | Guilherme Teles | Análise de qualidade e mapa de correlação |

---

## Síntese do Conteúdo

Este notebook realiza a análise exploratória completa do dataset PlantVillage, cobrindo:
1. Carregamento e estrutura do dataset
2. Distribuição de classes (balanceamento)
3. Análise visual de amostras por classe
4. Análise de canais de cor RGB
5. Análise de qualidade das imagens (nitidez)
6. Mapa de correlação espécie × doença
7. Preparação dos dados para treinamento

## 0. Instalação de Dependências

In [ ]:
# Instalar dependências necessárias (caso não estejam instaladas)
# !pip install tensorflow matplotlib seaborn pandas numpy opencv-python Pillow

## 1. Importações e Configurações

In [ ]:
# =============================================================================
# Identificação: Gabriel Vieira (10410264), Guilherme Rainho (10418251),
#                Guilherme Teles (10364065)
# Síntese: Importações e configurações globais do notebook
# Histórico:
#   22/05/2026 | Guilherme Rainho | Configuração inicial
# =============================================================================

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cv2
from PIL import Image
from pathlib import Path
from collections import defaultdict

# Configurações de visualização
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
sns.set_palette('husl')

# Seed para reprodutibilidade
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Caminho do dataset (ajuste conforme necessário)
DATASET_PATH = Path('../dataset/plantvillage')

print('✅ Importações concluídas!')
print(f'📁 Dataset path: {DATASET_PATH.resolve()}')

## 2. Carregamento e Estrutura do Dataset

In [ ]:
# =============================================================================
# Identificação: Gabriel Vieira (10410264), Guilherme Rainho (10418251),
#                Guilherme Teles (10364065)
# Síntese: Carregamento da estrutura do dataset e contagem de imagens por classe
# Histórico:
#   22/05/2026 | Gabriel Vieira | Implementação do carregamento
# =============================================================================

# Carregar estrutura do dataset
class_data = []

for class_dir in sorted(DATASET_PATH.iterdir()):
    if class_dir.is_dir():
        images = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.JPG')) + list(class_dir.glob('*.png'))
        class_name = class_dir.name
        # Extrair espécie e condição do nome da pasta (formato: Especie___Condicao)
        parts = class_name.split('___')
        species = parts[0].replace('_', ' ') if len(parts) > 0 else class_name
        condition = parts[1].replace('_', ' ') if len(parts) > 1 else 'Unknown'
        is_healthy = 'healthy' in condition.lower()
        
        class_data.append({
            'class_name': class_name,
            'species': species,
            'condition': condition,
            'is_healthy': is_healthy,
            'count': len(images),
            'path': class_dir
        })

df = pd.DataFrame(class_data)

# Estatísticas gerais
print('=' * 60)
print('📊 ESTATÍSTICAS GERAIS DO DATASET')
print('=' * 60)
print(f'Total de classes:    {len(df)}')
print(f'Total de imagens:    {df["count"].sum():,}')
print(f'Espécies únicas:     {df["species"].nunique()}')
print(f'Classes saudáveis:   {df["is_healthy"].sum()}')
print(f'Classes com doença:  {(~df["is_healthy"]).sum()}')
print(f'Média por classe:    {df["count"].mean():.0f} imagens')
print(f'Mínimo por classe:   {df["count"].min()} imagens ({df.loc[df["count"].idxmin(), "class_name"]})')
print(f'Máximo por classe:   {df["count"].max()} imagens ({df.loc[df["count"].idxmax(), "class_name"]})')
print('=' * 60)

## 3. Distribuição de Classes

In [ ]:
# =============================================================================
# Identificação: Gabriel Vieira (10410264), Guilherme Rainho (10418251),
#                Guilherme Teles (10364065)
# Síntese: Visualização da distribuição de imagens por classe
# Histórico:
#   22/05/2026 | Gabriel Vieira | Criação dos gráficos de distribuição
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Gráfico 1: Quantidade de imagens por classe (barras horizontais)
df_sorted = df.sort_values('count', ascending=True)
colors = ['#2ecc71' if h else '#e74c3c' for h in df_sorted['is_healthy']]

bars = axes[0].barh(range(len(df_sorted)), df_sorted['count'], color=colors, edgecolor='white', linewidth=0.5)
axes[0].set_yticks(range(len(df_sorted)))
axes[0].set_yticklabels([c.replace('___', '\n').replace('_', ' ') for c in df_sorted['class_name']], fontsize=7)
axes[0].set_xlabel('Número de Imagens')
axes[0].set_title('Distribuição de Imagens por Classe', fontweight='bold')
axes[0].axvline(df['count'].mean(), color='navy', linestyle='--', linewidth=1.5, label=f'Média ({df["count"].mean():.0f})')
patch_healthy = mpatches.Patch(color='#2ecc71', label='Saudável')
patch_disease = mpatches.Patch(color='#e74c3c', label='Doente')
axes[0].legend(handles=[patch_healthy, patch_disease, axes[0].lines[0]], loc='lower right', fontsize=9)

# Gráfico 2: Distribuição por espécie
species_count = df.groupby('species')['count'].sum().sort_values(ascending=False)
axes[1].bar(range(len(species_count)), species_count.values, color=sns.color_palette('husl', len(species_count)))
axes[1].set_xticks(range(len(species_count)))
axes[1].set_xticklabels(species_count.index, rotation=45, ha='right', fontsize=9)
axes[1].set_ylabel('Total de Imagens')
axes[1].set_title('Total de Imagens por Espécie', fontweight='bold')
for i, v in enumerate(species_count.values):
    axes[1].text(i, v + 50, str(v), ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('../docs/fig1_distribuicao_classes.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Figura salva em docs/fig1_distribuicao_classes.png')

## 4. Análise Visual — Amostras por Classe

In [ ]:
# =============================================================================
# Identificação: Gabriel Vieira (10410264), Guilherme Rainho (10418251),
#                Guilherme Teles (10364065)
# Síntese: Exibição de amostras aleatórias de cada classe para inspeção visual
# Histórico:
#   22/05/2026 | Guilherme Rainho | Criação da grade de amostras visuais
# =============================================================================

# Selecionar 12 classes aleatórias para visualização
sample_classes = df.sample(min(12, len(df)), random_state=SEED)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for idx, (_, row) in enumerate(sample_classes.iterrows()):
    # Pegar uma imagem aleatória da classe
    images = list(row['path'].glob('*.jpg')) + list(row['path'].glob('*.JPG'))
    if images:
        img_path = random.choice(images)
        img = Image.open(img_path).resize((224, 224))
        axes[idx].imshow(img)
        title = f"{row['species']}\n{row['condition']}"
        color = '#2ecc71' if row['is_healthy'] else '#e74c3c'
        axes[idx].set_title(title, fontsize=8, color=color, fontweight='bold')
    axes[idx].axis('off')

plt.suptitle('Amostras Visuais por Classe (aleatórias)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../docs/fig2_amostras_visuais.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Figura salva em docs/fig2_amostras_visuais.png')

## 5. Análise de Canais de Cor RGB

In [ ]:
# =============================================================================
# Identificação: Gabriel Vieira (10410264), Guilherme Rainho (10418251),
#                Guilherme Teles (10364065)
# Síntese: Análise estatística dos canais RGB por classe para identificar
#          padrões visuais característicos de cada condição
# Histórico:
#   22/05/2026 | Gabriel Vieira | Implementação da análise RGB
# =============================================================================

rgb_stats = []
SAMPLE_PER_CLASS = 30  # amostras por classe para análise RGB

print('🔍 Analisando canais RGB por classe...')
for _, row in df.iterrows():
    images = list(row['path'].glob('*.jpg')) + list(row['path'].glob('*.JPG'))
    sample = random.sample(images, min(SAMPLE_PER_CLASS, len(images)))
    
    r_means, g_means, b_means = [], [], []
    for img_path in sample:
        img = np.array(Image.open(img_path).resize((64, 64)))
        if img.ndim == 3 and img.shape[2] >= 3:
            r_means.append(img[:,:,0].mean())
            g_means.append(img[:,:,1].mean())
            b_means.append(img[:,:,2].mean())
    
    if r_means:
        rgb_stats.append({
            'class_name': row['class_name'],
            'species': row['species'],
            'condition': row['condition'],
            'is_healthy': row['is_healthy'],
            'R_mean': np.mean(r_means),
            'G_mean': np.mean(g_means),
            'B_mean': np.mean(b_means),
            'R_std': np.std(r_means),
            'G_std': np.std(g_means),
            'B_std': np.std(b_means),
        })

df_rgb = pd.DataFrame(rgb_stats)
print(f'✅ Análise RGB concluída para {len(df_rgb)} classes')

# Comparar médias RGB entre plantas saudáveis e doentes
print('\n📊 Médias RGB — Saudável vs Doente:')
print(df_rgb.groupby('is_healthy')[['R_mean', 'G_mean', 'B_mean']].mean().rename(index={True: 'Saudável', False: 'Doente'}))

In [ ]:
# Visualizar distribuição RGB
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
channels = [('R_mean', '#e74c3c', 'Canal R (Vermelho)'),
            ('G_mean', '#2ecc71', 'Canal G (Verde)'),
            ('B_mean', '#3498db', 'Canal B (Azul)')]

for ax, (col, color, title) in zip(axes, channels):
    healthy = df_rgb[df_rgb['is_healthy']][col]
    diseased = df_rgb[~df_rgb['is_healthy']][col]
    ax.hist(healthy, bins=15, alpha=0.7, color='#2ecc71', label='Saudável', edgecolor='white')
    ax.hist(diseased, bins=15, alpha=0.7, color='#e74c3c', label='Doente', edgecolor='white')
    ax.set_xlabel('Valor Médio do Canal (0–255)')
    ax.set_ylabel('Frequência (classes)')
    ax.set_title(title, fontweight='bold')
    ax.legend()
    ax.axvline(healthy.mean(), color='#27ae60', linestyle='--', linewidth=1.5)
    ax.axvline(diseased.mean(), color='#c0392b', linestyle='--', linewidth=1.5)

plt.suptitle('Distribuição dos Canais RGB — Saudável vs Doente', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/fig3_analise_rgb.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Figura salva em docs/fig3_analise_rgb.png')

## 6. Análise de Qualidade das Imagens (Nitidez)

In [ ]:
# =============================================================================
# Identificação: Gabriel Vieira (10410264), Guilherme Rainho (10418251),
#                Guilherme Teles (10364065)
# Síntese: Detecção de imagens de baixa qualidade usando operador Laplaciano.
#          Imagens com variância < 100 são consideradas borradas/baixa qualidade.
# Histórico:
#   22/05/2026 | Guilherme Teles | Implementação da análise de nitidez
# =============================================================================

SHARPNESS_THRESHOLD = 100  # Limiar de variância do Laplaciano
SAMPLE_FOR_QUALITY = 20    # Imagens amostradas por classe para análise de qualidade

quality_stats = []
print('🔍 Analisando qualidade das imagens...')

for _, row in df.iterrows():
    images = list(row['path'].glob('*.jpg')) + list(row['path'].glob('*.JPG'))
    sample = random.sample(images, min(SAMPLE_FOR_QUALITY, len(images)))
    
    sharpness_values = []
    low_quality_count = 0
    
    for img_path in sample:
        img_gray = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2GRAY)
        variance = cv2.Laplacian(img_gray, cv2.CV_64F).var()
        sharpness_values.append(variance)
        if variance < SHARPNESS_THRESHOLD:
            low_quality_count += 1
    
    quality_stats.append({
        'class_name': row['class_name'],
        'species': row['species'],
        'is_healthy': row['is_healthy'],
        'mean_sharpness': np.mean(sharpness_values),
        'low_quality_pct': low_quality_count / len(sample) * 100
    })

df_quality = pd.DataFrame(quality_stats)
total_low_quality = df_quality['low_quality_pct'].mean()

print(f'✅ Análise de qualidade concluída!')
print(f'   Percentual médio de imagens de baixa qualidade: {total_low_quality:.1f}%')
print(f'   Classes com > 10% de imagens de baixa qualidade: {(df_quality["low_quality_pct"] > 10).sum()}')

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(df_quality['mean_sharpness'], bins=20, color='#3498db', edgecolor='white')
axes[0].axvline(SHARPNESS_THRESHOLD, color='red', linestyle='--', linewidth=2, label=f'Limiar ({SHARPNESS_THRESHOLD})')
axes[0].set_xlabel('Variância do Laplaciano (nitidez)')
axes[0].set_ylabel('Número de Classes')
axes[0].set_title('Distribuição de Nitidez por Classe', fontweight='bold')
axes[0].legend()

axes[1].barh(df_quality.nlargest(10, 'low_quality_pct')['class_name'].str.replace('___', '\n'),
             df_quality.nlargest(10, 'low_quality_pct')['low_quality_pct'],
             color='#e74c3c', edgecolor='white')
axes[1].set_xlabel('% Imagens de Baixa Qualidade')
axes[1].set_title('Top 10 Classes com Mais Imagens de Baixa Qualidade', fontweight='bold')
axes[1].axvline(10, color='orange', linestyle='--', linewidth=1.5, label='Limite 10%')
axes[1].legend()

plt.tight_layout()
plt.savefig('../docs/fig4_qualidade_imagens.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Figura salva em docs/fig4_qualidade_imagens.png')

## 7. Mapa de Correlação Espécie × Doença

In [ ]:
# =============================================================================
# Identificação: Gabriel Vieira (10410264), Guilherme Rainho (10418251),
#                Guilherme Teles (10364065)
# Síntese: Mapa de calor de co-ocorrência espécie × condição,
#          mostrando quais espécies têm mais variações de doença
# Histórico:
#   22/05/2026 | Guilherme Teles | Criação do mapa de correlação
# =============================================================================

# Criar pivot table: espécie × condição (quantidade de imagens)
pivot = df.pivot_table(index='species', columns='condition', values='count', fill_value=0)

plt.figure(figsize=(18, 8))
sns.heatmap(
    pivot,
    annot=True, fmt='d', cmap='YlOrRd',
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'Número de Imagens'}
)
plt.title('Mapa de Co-ocorrência: Espécie × Condição', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Condição', fontsize=11)
plt.ylabel('Espécie', fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('../docs/fig5_correlacao_especie_doenca.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Figura salva em docs/fig5_correlacao_especie_doenca.png')

# Participação do tomate no dataset
tomato_total = df[df['species'] == 'Tomato']['count'].sum()
total = df['count'].sum()
print(f'\n🍅 Participação do tomate no dataset: {tomato_total/total*100:.1f}% ({tomato_total:,} imagens)')
print(f'   Número de classes de tomate: {len(df[df["species"] == "Tomato"])}')

## 8. Preparação dos Dados para Treinamento

In [ ]:
# =============================================================================
# Identificação: Gabriel Vieira (10410264), Guilherme Rainho (10418251),
#                Guilherme Teles (10364065)
# Síntese: Pipeline de preparação dos dados com divisão treino/val/teste
#          e configuração do data augmentation
# Histórico:
#   22/05/2026 | Guilherme Rainho | Implementação do pipeline de dados
# =============================================================================

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Parâmetros
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
TRAIN_SPLIT = 0.70
VAL_SPLIT = 0.15
# TEST_SPLIT = 0.15 (implícito)

# Data generators com augmentation para treino
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest',
    validation_split=1 - TRAIN_SPLIT  # reservar 30% para val+test
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

print('✅ Pipeline de preparação configurado!')
print(f'   Tamanho de entrada: {IMG_SIZE}')
print(f'   Batch size: {BATCH_SIZE}')
print(f'   Divisão: {TRAIN_SPLIT*100:.0f}% treino / {VAL_SPLIT*100:.0f}% validação / {(1-TRAIN_SPLIT-VAL_SPLIT)*100:.0f}% teste')
print('\n📋 Augmentation configurado:')
print('   ✔ Rotação: ±30°')
print('   ✔ Deslocamento horizontal/vertical: ±10%')
print('   ✔ Zoom: ±20%')
print('   ✔ Flip horizontal e vertical')
print('   ✔ Variação de brilho: 80%–120%')

## 9. Resumo e Conclusões da Análise Exploratória

### Principais achados:

| Aspecto | Achado | Implicação |
|---------|--------|------------|
| **Balanceamento** | 78% das classes têm 1.000–2.000 imagens; 11 classes < 1.000 | Aplicar class weights no treinamento |
| **Canal Verde (G)** | Folhas doentes têm média de G significativamente menor | CNN pode explorar esse padrão para discriminação |
| **Canal Vermelho (R)** | Folhas com estresse hídrico têm R médio mais alto | Indicador de clorose e necrose |
| **Qualidade** | ~2,3% das imagens têm nitidez abaixo do limiar | Filtrar antes do treinamento |
| **Dominância do tomate** | ~40% do dataset é de tomate (18/38 classes) | Avaliar generalização nas demais culturas |
| **MobileNetV2 baseline** | 78,4% de acurácia em 10 épocas | Arquitetura viável; espera-se > 85% com fine-tuning completo |

### Próximos passos:
1. Fine-tuning completo do MobileNetV2 (50+ épocas)
2. Avaliação por espécie individualmente
3. Conversão para TensorFlow Lite com quantização int8
4. Teste de inferência no Raspberry Pi 4
5. Integração com pipeline de comunicação LoRaWAN/MQTT